# Notebook 01 — ETL Amazon Reviews

**Salidas:** `data/clean/ratings.csv` y `data/clean/sentiment.csv`


## 0. Conexión al clúster Spark

In [30]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.sql.functions import (
    col, lower, trim, length, regexp_replace,
    when, count
)

spark = SparkSession.builder.appName("TFM_ETL_Amazon").getOrCreate()

## 1. Carga del dataset 

In [31]:
# Ruta dentro del contenedor
RAW_PATH   = "/opt/spark-data/data/raw/Reviews.csv"
CLEAN_PATH = "/opt/spark-data/data/clean/"

df_raw = spark.read.csv(RAW_PATH,  header=True)

print(f"Filas totales: {df_raw.count():,}")
print(f"Columnas:      {df_raw.columns}")
df_raw.printSchema()

Filas totales: 568,454
Columnas:      ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']
root
 |-- Id: string (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: string (nullable = true)
 |-- HelpfulnessDenominator: string (nullable = true)
 |-- Score: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)



## 2. Análisis de nulos 

In [32]:
# Conteo de nulos por columna
null_counts = df_raw.select([
    count(when(col(c).isNull() | (F.col(c) == ""), c)).alias(c)
    for c in df_raw.columns
])
null_counts.show()

+---+---------+------+-----------+--------------------+----------------------+-----+----+-------+----+
| Id|ProductId|UserId|ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|Time|Summary|Text|
+---+---------+------+-----------+--------------------+----------------------+-----+----+-------+----+
|  0|        0|     0|          0|                   2|                     2|    3|   5|      6|  10|
+---+---------+------+-----------+--------------------+----------------------+-----+----+-------+----+



Antes de limpiar hay que cuantificar el problema. Se detectan tanto null como cadenas vacías "" porque ambos representan ausencia de dato en un CSV. El resultado muestra que el dataset está muy limpio (máximo 10 nulos en Text), lo que justifica la estrategia de eliminación directa en el siguiente paso en lugar de imputación.

## 3. Limpieza general

In [33]:
REQUIRED_COLS = ["UserId", "ProductId", "Score", "Text"]

n_original = df_raw.count()

# Eliminar nulos en columnas críticas
df_clean = df_raw.dropna(subset=REQUIRED_COLS)
n_after_null = df_clean.count()
print(f"Eliminados por nulos: {n_original - n_after_null:,} filas")

# Filtrar ratings fuera de rango
df_clean = df_clean.filter(
    (col("Score").cast("int") >= 1) &
    (col("Score").cast("int") <= 5)
)
n_after_rating = df_clean.count()
print(f"Eliminados por rating inválido: {n_after_null - n_after_rating:,} filas")

# Eliminar reseñas demasiado cortas (menos de 10 caracteres)
df_clean = df_clean.filter(length(trim(col("Text"))) > 10)
n_after_length = df_clean.count()
print(f"Eliminados por texto muy corto: {n_after_rating - n_after_length:,} filas")

print(f"Total final: {n_after_length:,} filas")

Eliminados por nulos: 10 filas
Eliminados por rating inválido: 1,641 filas
Eliminados por texto muy corto: 438 filas
Total final: 566,365 filas


Se aplican tres filtros en cascada, guardando el conteo tras cada uno para tener trazabilidad:
- Nulos en columnas críticas: sin UserId, ProductId, Score o Text la fila no aporta valor a ninguno de los tres modelos.
- Rating fuera de rango: el dataset Amazon usa escala 1-5; valores fuera de ese rango indican filas corruptas. El cast a int convierte strings inválidos en null, que el filtro >= 1 elimina automáticamente.
- Texto muy corto: reseñas de menos de 10 caracteres no contienen información semántica útil para el modelo de sentimiento ni para el análisis.

## 4. Normalización de texto

In [34]:
df_clean = df_clean \
    .withColumn("text_clean",
        regexp_replace(
            regexp_replace(
                lower(col("Text")),
                r"<[^>]+>",      
                " "
            ),
            r"[^a-z\s]",        
            " "
        )
    ) \
    .withColumn("text_clean", regexp_replace(col("text_clean"), r"\s+", " ")) \
    .withColumn("text_clean", trim(col("text_clean")))

# Muestra antes/después
df_clean.select("Text", "text_clean").show(3, truncate=80)

+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            Text|                                                                      text_clean|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|I have bought several of the Vitality canned dog food products and have found...|i have bought several of the vitality canned dog food products and have found...|
|"Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually ...|product arrived labeled as jumbo salted peanuts the peanuts were actually sma...|
|"This is a confection that has been around a few centuries.  It is a light, p...|this is a confection that has been around a few centuries it is a light pillo...|
+---------------

El pipeline requiere texto normalizado. Se aplican cuatro transformaciones en orden:
1. lower(): tranforma el texto a minúsculas.
2. regexp_replace HTML: algunas reseñas contienen etiquetas copiadas de páginas web.
3. regexp_replace no-letras: elimina puntuación, números y caracteres especiales.
4. Colapsar espacios + trim(): los pasos anteriores generan espacios múltiples, se normalizan a uno solo y se eliminan los extremos. El resultado se guarda en una nueva columna text_clean.

## 5. Generación del dataset de sentiment

In [35]:
# Etiqueta binaria: 1=positivo (4-5 estrellas), 0=negativo (1-2 estrellas)
# Descartamos 3 estrellas (neutro ambiguo)
df_sentiment = df_clean \
    .filter(col("Score").cast("int") != 3) \
    .withColumn(
        "sentiment",
        when(col("Score").cast("int") >= 4, 1).otherwise(0)
    ) \
    .select("UserId", "ProductId", "Score", "text_clean", "sentiment")

# Balance de clases
print("Balance de clases (sentiment):")
df_sentiment.groupBy("sentiment").count().show()

# Guardar
df_sentiment.toPandas().to_csv(f"{CLEAN_PATH}sentiment.csv", index=False)

print(f"Guardado: {CLEAN_PATH}sentiment.csv")

Balance de clases (sentiment):
+---------+------+
|sentiment| count|
+---------+------+
|        1|441703|
|        0| 82223|
+---------+------+

Guardado: /opt/spark-data/data/clean/sentiment.csv


Se construye la variable objetivo para el modelo de sentimiento:
- Eliminar Score=3 (neutro): las reseñas de 3 estrellas son ambiguas.
- Etiqueta binaria 0/1: la clasificación binaria (positivo/negativo) es más robusta.
- `toPandas().to_csv()`: se usa pandas para el guardado.

## 6. Generación del dataset de ratings

In [36]:
df_ratings_raw = df_clean.select(
    col("UserId"),
    col("ProductId"),
    col("Score").cast("float").alias("rating"),
    col("Time").cast("long").alias("timestamp") 
)

# Filtrar: usuarios con >= 5 reseñas
user_counts = df_ratings_raw.groupBy("UserId") \
    .agg(count("*").alias("n_reviews"))

active_users = user_counts.filter(col("n_reviews") >= 5).select("UserId")

# Filtrar: productos con >= 3 reseñas
item_counts = df_ratings_raw.groupBy("ProductId") \
    .agg(count("*").alias("n_reviews"))

active_items = item_counts.filter(col("n_reviews") >= 3).select("ProductId")

df_ratings = df_ratings_raw \
    .join(active_users, on="UserId", how="inner") \
    .join(active_items, on="ProductId", how="inner")

print(f"Usuarios activos: {active_users.count():,}")
print(f"Ítems activos:    {active_items.count():,}")
print(f"Interacciones:    {df_ratings.count():,}")

# Guardar
df_ratings.toPandas().to_csv(f"{CLEAN_PATH}ratings.csv", index=False)

print(f"Guardado: {CLEAN_PATH}ratings.csv")

Usuarios activos: 23,496
Ítems activos:    31,542
Interacciones:    227,774
Guardado: /opt/spark-data/data/clean/ratings.csv


Se construye la matriz de interacciones para el motor de recomendación:
- `timestamp`: se añade la columna `Time`.
- Filtro usuarios ≥ 5 reseñas: usuarios con menos interacciones no aportan suficiente señal para que el filtrado calcule similitudes fiables entre ítems.
- Filtro ítems ≥ 3 reseñas: productos con muy pocas valoraciones generan columnas casi vacías en la matriz usuario-ítem.

## 7. Resumen del ETL

In [37]:
print("=" * 50)
print("RESUMEN ETL AMAZON REVIEWS")
print("=" * 50)
print(f"Filas originales:          {n_original:>10,}")
print(f"Tras eliminar nulos:       {n_after_null:>10,}")
print(f"Tras filtrar ratings:      {n_after_rating:>10,}")
print(f"Tras filtrar texto corto:  {n_after_length:>10,}")
print(f"Dataset sentimiento:       {df_sentiment.count():>10,}")
print(f"Dataset ratings:           {df_ratings.count():>10,}")
print("=" * 50)

spark.stop()

RESUMEN ETL AMAZON REVIEWS
Filas originales:             568,454
Tras eliminar nulos:          568,444
Tras filtrar ratings:         566,803
Tras filtrar texto corto:     566,365
Dataset sentimiento:          523,926
Dataset ratings:              227,774
